In [1]:
import pandas as pd
import numpy as np
import re
import string

In [2]:
fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")

print("Fake shape:", fake.shape)
print("True shape:", true.shape)

Fake shape: (23481, 4)
True shape: (21417, 4)


In [3]:
fake["label"] = 0   # Fake news
true["label"] = 1   # Real news

data = pd.concat([fake, true], axis=0)
data = data.sample(frac=1).reset_index(drop=True)

print(data["label"].value_counts())

label
0    23481
1    21417
Name: count, dtype: int64


In [4]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\n', '', text)
    text = re.sub(r'\w*\d\w*', '', text)
    return text

data["clean_text"] = data["text"].apply(clean_text)

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(data["clean_text"])
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression()
model.fit(X_train, y_train)

print("Accuracy:", model.score(X_test, y_test))

Accuracy: 0.9865256124721603


In [6]:
def predict_news(text):
    text_clean = clean_text(text)
    text_vector = vectorizer.transform([text_clean])
    prediction = model.predict(text_vector)[0]

    if prediction == 1:
        print("Real News ✅")
    else:
        print("Fake News ❌")

In [7]:
predict_news("""
The U.S. Senate passed a new infrastructure bill after months of negotiation.
The bill is expected to improve transportation and create jobs.
""")

Fake News ❌


In [8]:
predict_news("""
Scientists confirm that dinosaurs are living inside the Earth's core.
""")

Fake News ❌


In [9]:
print(true["text"].iloc[0])

WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called himself a “fiscal conservative” on Sunday and urged budget restraint in 2018. In keeping with a sharp pivot under way among Republicans, U.S. Representative Mark Meadows, speaking on CBS’ “Face the Nation,” drew a hard line on federal spending, which lawmakers are bracing to do battle over in January. When they return from the holidays on Wednesday, lawmakers will begin trying to pass a federal budget in a fight likely to be linked to other issues, such as immigration policy, even as the November congressional election campaigns approach in which Republicans will seek to keep control of Congress. President Donald Trump and his Republicans want a big budget increase in military spending, while Democrats also want proportional increases for non-defense “discretionary” spending on programs that support educati

In [10]:
predict_news(""" WASHINGTON (Reuters) - A federal judge in Seattle partially blocked U.S. President Donald Trump’s newest restrictions on refugee admissions on Saturday, the latest legal defeat for his efforts to curtail immigration and travel to the United States. The decision by U.S. District Judge James Robart is the first judicial curb on rules the Trump administration put into place in late October that have contributed significantly to a precipitous drop in the number of refugees being admitted into the country. Refugees and groups that assist them argued in court that the administration’s policies violated the Constitution and federal rulemaking procedures, among other claims. Department of Justice attorneys argued in part that U.S. law grants the executive branch the authority to limit refugee admissions in the way that it had done so. On Oct. 24, the Trump administration effectively paused refugee admissions from 11 countries mostly in the Middle East and Africa, pending a 90-day security review, which was set to expire in late January. The countries subject to the review are Egypt, Iran, Iraq, Libya, Mali, North Korea, Somalia, South Sudan, Sudan, Syria and Yemen. For each of the last three years, refugees from the 11 countries made up more than 40 percent of U.S. admissions. A Reuters review of State Department data showed that as the review went into effect, refugee admissions from the 11 countries plummeted. Robart ruled that the administration could carry out the security review, but that it could not stop processing or admitting refugees from the 11 countries in the meantime, as long as those refugees have a “bona fide” connection to the United States. As part of its new restrictions, the Trump administration had also paused a program that allowed for family reunification for refugees, pending further security screening procedures being put into place. Robart ordered the government to re-start the program, known as “follow-to-join”. Approximately 2,000 refugees were admitted into the United States in fiscal year 2015 under the program, according to Department of Homeland Security data. Refugee advocacy groups praised Robart’s decision.  “This ruling brings relief to thousands of refugees in precarious situations in the Middle East and East Africa, as well as to refugees already in the U.S. who are trying to reunite with their spouses and children,” said Mariko Hirose, litigation director for the International Refugee Assistance Project, one of the plaintiffs in the case. A Justice Department spokeswoman, Lauren Ehrsam, said the department disagrees with Robart’s ruling and is “currently evaluating the next steps”. Robart, who was appointed to the bench by Republican former President George W. Bush, emerged from relative obscurity in February, when he issued a temporary order to lift the first version of Trump’s travel ban. On Twitter, Trump called him a “so-called judge” whose “ridiculous” opinion “essentially takes law-enforcement away from our country”. Robart’s ruling represented the second legal defeat in two days for the Trump administration. On Friday, a U.S. appeals court said Trump’s travel ban targeting six Muslim-majority countries should not be applied to people with strong U.S. ties, but said its ruling would be put on hold pending a decision by the U.S. Supreme Court.  """)

Real News ✅


In [12]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [13]:
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(data["clean_text"])

X_seq = tokenizer.texts_to_sequences(data["clean_text"])
X_pad = pad_sequences(X_seq, maxlen=200)

y = data["label"]

In [14]:
from sklearn.model_selection import train_test_split

X_train_dl, X_test_dl, y_train_dl, y_test_dl = train_test_split(
    X_pad, y, test_size=0.2, random_state=42
)

In [15]:
model_dl = Sequential()

model_dl.add(Embedding(input_dim=10000, output_dim=128, input_length=200))
model_dl.add(LSTM(64))
model_dl.add(Dense(1, activation='sigmoid'))

model_dl.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [16]:
history = model_dl.fit(
    X_train_dl,
    y_train_dl,
    epochs=3,
    batch_size=64,
    validation_data=(X_test_dl, y_test_dl)
)

Epoch 1/3
562/562 ━━━━━━━━━━━━━━━━━━━━ 140s 242ms/step - accuracy: 0.9179 - loss: 0.2225 - val_accuracy: 0.7830 - val_loss: 0.4156
Epoch 2/3
562/562 ━━━━━━━━━━━━━━━━━━━━ 137s 244ms/step - accuracy: 0.9384 - loss: 0.1514 - val_accuracy: 0.9840 - val_loss: 0.0538
Epoch 3/3
562/562 ━━━━━━━━━━━━━━━━━━━━ 137s 244ms/step - accuracy: 0.9920 - loss: 0.0295 - val_accuracy: 0.9872 - val_loss: 0.0404


In [17]:
def predict_news_dl(text):
    text_clean = clean_text(text)
    seq = tokenizer.texts_to_sequences([text_clean])
    pad = pad_sequences(seq, maxlen=200)

    prediction = model_dl.predict(pad)[0][0]

    if prediction > 0.5:
        print("Real News ✅")
    else:
        print("Fake News ❌")

    print("Confidence:", round(float(prediction), 3))

In [18]:
predict_news_dl("""
WASHINGTON (Reuters) - The U.S. Senate passed a new economic reform bill after months of debate.
The legislation aims to strengthen infrastructure and improve job opportunities.
""")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 512ms/step
Real News ✅
Confidence: 0.999


In [19]:
predict_news_dl("""
WASHINGTON (Reuters) - The U.S. Senate passed a new economic reform bill after months of debate.
The legislation aims to strengthen infrastructure and improve job opportunities.
""")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
Real News ✅
Confidence: 0.999


In [20]:
predict_news_dl("""
Aliens have officially taken control of the White House.
""")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step
Fake News ❌
Confidence: 0.016
